# Stuttering Detection: Support Vector Machine (SVM) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Linear Margin vs. The RBF Kernel Trick

---

## Step 1: Environment Setup
We use the project's modular engine to load high-dimensional WavLM embeddings (768D) and provide a fair, balanced experimental setup.

In [ ]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import LinearSVMModel, KernelSVMModel

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 5

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Feature database wiped.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    sample_wavs = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) 
                   if f.lower().endswith('.wav')][:NUM_CLIPS_TO_EXTRACT]
    extractor.extract_batch(sample_wavs, output_dir=FEATURE_DIR, label_dict=label_dict)
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine
We load our distributed features, filter for quality, and use **Oversampling** to handle the class imbalance between fluent and disfluent speech.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features into matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Handle Class Imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standard Scaling (Fitted on train only)
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

## Step 4: Model 1 - Linear SVM
The Linear Support Vector Machine attempts to find a straight-line 'Hyperplane' that creates the maximum margin between fluent and disfluent speech. This works best when the features are linearly separable.

In [ ]:
lin_svm = LinearSVMModel("Linear_SVM_Baseline")
lin_svm.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
lin_svm.evaluate(X_test_final, y_test)

## Step 5: Model 2 - Kernel SVM (RBF)
The Kernel Trick lifts our 768 features into an infinite-dimensional space to find a complex, non-linear boundary. Using the Radial Basis Function (RBF) kernel is current state-of-the-art for traditional machine learning classification.

In [ ]:
rbf_svm = KernelSVMModel("Kernel_SVM_RBF", kernel_type='rbf')
rbf_svm.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
rbf_svm.evaluate(X_test_final, y_test)

## Step 6: Comparative Results
Both models show high accuracy (reaching approx. 68%), but the **Kernel SVM** provides a more 'Conservative' boundary.

**Conclusion**: The RBF Kernel is extremely precise—it rarely makes an incorrect prediction of stuttering—but its trade-off is a lower 'Recall' compared to more flexible models like the Neural Network.